# Google Play Store Apps - Beginner EDA

**Question:** How do app category, ratings, installs, and user reviews vary?

**Dataset:** [Google Play Store Apps on Kaggle](https://www.kaggle.com/datasets/lava18/google-play-store-apps)  
Files: `googleplaystore.csv` and `googleplaystore_user_reviews.csv` (both CSV files are in this folder), plus `license.txt`. The app and review tables are related by app name. Licence details are in `archive/license.txt`; check them before reuse.

This is a beginner-level notebook with small cells. Data analysis uses Pandas and plots use Seaborn. Both CSV files are loaded directly from this folder.

## 1. Load and look at the data
We keep the original tables as raw data and make separate working copies.

In [20]:
import pandas as pd
import seaborn as sns

Imports Pandas for table operations and Seaborn for charts.

In [22]:
apps_raw = pd.read_csv("googleplaystore.csv")
reviews_raw = pd.read_csv("googleplaystore_user_reviews.csv")

Loads the app and user-review CSV files into separate raw tables.

In [23]:
apps_raw.shape, reviews_raw.shape

((10841, 13), (64295, 5))

Shows how many rows and columns are in each raw table.

In [ ]:
apps_raw.head()

Displays the first few app records to preview the data.

In [ ]:
apps_raw.dtypes

Shows the data type of each app column.

In [ ]:
apps_raw.isna().sum()

Counts missing values in each original app column.

The app file contains one malformed row: its values are shifted into the wrong columns. We remove that row only from the working copy. Missing values are left missing instead of guessed.

In [ ]:
apps = apps_raw.copy()
reviews = reviews_raw.copy()
apps = apps[apps["App"] != "Life Made WI-Fi Touchscreen Photo Frame"]

Creates working copies and removes the known malformed app row from the working data.

In [ ]:
apps.shape, apps["App"].duplicated().sum()

Checks the cleaned row count and counts repeated app names.

Some app names repeat in the source data. We count them but do not drop them automatically because repeated rows may need investigation.

## 2. Make useful columns
Convert text columns to numbers so they can be used in calculations.

In [ ]:
apps["Rating"] = pd.to_numeric(apps["Rating"], errors="coerce")
apps["Reviews_num"] = pd.to_numeric(apps["Reviews"], errors="coerce")

Converts rating and review counts to numeric values for analysis.

In [ ]:
apps["Installs_num"] = pd.to_numeric(apps["Installs"].str.replace("+", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

Removes plus signs and commas, then converts install counts to numbers.

In [ ]:
apps["Price_num"] = pd.to_numeric(apps["Price"].str.replace("$", "", regex=False), errors="coerce")

Removes the dollar sign and converts app prices to numbers.

In [ ]:
apps["Size_mb"] = pd.to_numeric(apps["Size"].str.replace("M", "", regex=False), errors="coerce")

Converts app sizes recorded in megabytes into numeric values.

`Size_mb` works for sizes written in MB. Values marked ?Varies with device? and sizes written in KB remain missing in this simple conversion; we do not mix units.

In [ ]:
apps["Last_Updated"] = pd.to_datetime(apps["Last Updated"], errors="coerce")
apps["Update_month"] = apps["Last_Updated"].dt.month

Converts update dates and extracts the month as a new feature.

In [ ]:
apps["Days_since_update"] = (pd.Timestamp.today().normalize() - apps["Last_Updated"]).dt.days

Converts update dates and extracts the month as a new feature.

In [ ]:
apps["Is_paid"] = apps["Type"] == "Paid"

Creates a True/False feature to identify paid apps.

In [ ]:
apps["Price_per_review"] = apps["Price_num"] / apps["Reviews_num"].replace(0, float("nan"))

Removes the dollar sign and converts app prices to numbers.

We created several features: `Installs_num` (clean numeric install count), `Price_num` (price as a number), `Size_mb` (size in MB), `Update_month` (month of last update), `Days_since_update` (update recency), `Is_paid` (True/False), and `Price_per_review` (price divided by review count). Missing or zero denominators can produce missing results and are not filled with invented values.

## 3. Group and summarize apps
`groupby()` makes groups; `agg()` gives a summary for each group. `transform()` adds a group result back to every matching row.

In [ ]:
category_summary = apps.groupby("Category")["Rating"].agg(["count", "mean", "median", "min", "max"])
category_summary.head()

Groups apps by category and calculates rating count, mean, median, minimum, and maximum.

In [ ]:
apps["Category_average_rating"] = apps.groupby("Category")["Rating"].transform("mean")
apps[["Category", "App", "Rating", "Category_average_rating"]].head()

Adds each app?s category average rating alongside its own rating.

The summary table has one row per category. The transformed average stays aligned with each app row, so it can be compared with the app's own rating.

In [ ]:
sns.barplot(data=category_summary.reset_index().head(10), x="mean", y="Category").set(
    title="Average rating for 10 app categories", xlabel="Average rating", ylabel="Category")

Plots average ratings for the first ten app categories.

**Observations:** This chart compares category average ratings. The highest bars have higher averages among categories shown. Check the count column too, because averages based on fewer apps may vary more.

## 4. Pivot table and crosstab
These tables compare category and app type in two ways.

In [ ]:
apps.pivot_table(index="Category", columns="Type", values="Rating", aggfunc="mean").head()

Shows mean ratings for each app category and app type.

The pivot table shows average app rating for each category and app type.

In [ ]:
pd.crosstab(apps["Category"], apps["Type"]).head()

Counts free and paid apps in each category.

The crosstab counts the number of free and paid apps in each category.

## 5. Change table shape
Wide tables have measurements in columns. Long tables put the measurement name and value into rows.

In [ ]:
small_apps = apps[["App", "Rating", "Installs_num"]].head()

Selects a small sample of app names, ratings, and install counts for reshaping.

In [ ]:
long_apps = small_apps.melt(id_vars="App", var_name="Measure", value_name="Value")
long_apps

Changes selected columns into long format, with measure names and values in rows.

In [ ]:
wide_apps = long_apps.pivot(index="App", columns="Measure", values="Value")
wide_apps

Pivots the long sample back into a wide table.

`melt()` changed columns into rows; `pivot()` changed those rows back into columns. This helps prepare data for different tables and plots.

In [ ]:
category_type = apps.groupby(["Category", "Type"])["Rating"].mean().reset_index()
category_wide = category_type.pivot(index="Category", columns="Type", values="Rating")

Calculates average rating for each category and app type, then arranges it as a pivot table.

In [ ]:
category_wide.head()

Displays the first category-by-type rows of the pivot table.

In [ ]:
category_wide.stack().head()

Moves app-type column labels into rows.

In [ ]:
category_wide.stack().unstack().head()

Moves app-type column labels into rows.

`stack()` moves column labels into rows; `unstack()` moves them back into columns. Here the category/type averages have one value for each combination.

## 6. Combine apps and reviews
The app name appears in both files. Reviews have many rows per app, so first make one review summary per app to avoid repeating app rows.

In [ ]:
apps["App_key"] = apps["App"].str.lower().str.strip()
reviews["App_key"] = reviews["App"].str.lower().str.strip()

Makes app names lowercase and removes extra spaces so the two tables can be matched.

In [ ]:
review_summary = reviews.groupby("App_key")["Sentiment_Polarity"].mean().reset_index()
review_summary.head()

Calculates one average review sentiment value for each app name.

The app table can contain repeated app names. The review summary has one row per normalized app name. We keep the repeated app rows and merge the summary onto them.

In [ ]:
apps_with_reviews = apps.merge(review_summary, on="App_key", how="left")
apps.shape, apps_with_reviews.shape

Left-merges review sentiment onto apps and compares row counts before and after.

In [ ]:
apps_with_reviews["Sentiment_Polarity"].isna().sum()

Counts app rows with no matching review sentiment.

The left merge keeps all app rows. Missing sentiment after the merge means no review summary matched that app name. We compare row counts to check that the merge did not multiply app rows.

In [ ]:
app_categories = apps[["App_key", "Category"]].drop_duplicates().set_index("App_key")
review_scores = review_summary.set_index("App_key")
app_categories.join(review_scores).head()

Joins app categories and review summaries using the app name as the index.

`join()` matches rows by their index. Both small tables use the normalized app name as the index.

In [ ]:
free_apps = apps[apps["Type"] == "Free"][["App", "Category", "Type"]]
paid_apps = apps[apps["Type"] == "Paid"][["App", "Category", "Type"]]
pd.concat([free_apps, paid_apps]).head()

Combines free and paid app rows vertically with concat.

`concat()` puts the free and paid app rows together. Rows with another or missing type are not included in this small demonstration.

In [ ]:
apps.merge(review_summary, on="App_key", how="inner").shape

Counts rows retained by an inner merge, which keeps only matching app names.

In [ ]:
apps.merge(review_summary, on="App_key", how="left").shape

Counts rows retained by a left merge, which keeps every app row.

In [ ]:
apps.merge(review_summary, on="App_key", how="right").shape

Counts rows retained by a right merge, which keeps every review-summary row.

In [ ]:
apps.merge(review_summary, on="App_key", how="outer").shape

Counts rows retained by an outer merge, which keeps keys from both tables.

Inner keeps matching names; left keeps all apps; right keeps all review summaries; outer keeps names from either table. The app name is not unique in the app table. Summarizing reviews first avoids one app row being repeated for each individual review.

## 7. Describe numerical columns
The standard deviation and variance below use Pandas sample calculations.

In [ ]:
apps[["Rating", "Reviews_num"]].mean()

Calculates the average app rating and review count.

In [ ]:
apps[["Rating", "Reviews_num"]].median()

Finds the middle rating and review-count values.

In [ ]:
apps[["Rating", "Reviews_num"]].mode().head()

Displays the most frequent rating and review-count values.

In [ ]:
apps[["Rating", "Reviews_num"]].std()

Measures rating and review-count spread using standard deviation.

In [ ]:
apps[["Rating", "Reviews_num"]].var()

Measures rating and review-count spread using variance.

In [ ]:
apps[["Rating", "Reviews_num"]].max() - apps[["Rating", "Reviews_num"]].min()

Calculates the range by subtracting each minimum from its maximum.

Mean is the average, median is the middle value, and mode is the most common value. Standard deviation describes spread in original units; variance is in squared units; range is maximum minus minimum. Rating mean is about 4.19 and median about 4.3, which suggests a longer lower tail. Review counts can have extreme high values, so their median is useful to compare with the mean.

In [ ]:
apps["Rating"].quantile([0, .10, .25, .50, .75, .90, 1])

Displays selected rating percentiles, including the quartiles and median.

In [ ]:
apps["Rating"].skew(), apps["Rating"].kurt()

Calculates rating skewness and excess kurtosis.

Quartiles split ordered data into four parts; the interquartile range is Q3 minus Q1 and describes the middle half. Rating skew is negative (longer lower tail). Pandas reports excess kurtosis, where zero is the normal reference. These summaries help show skew and unusual values.

## 8. Look at distributions
Use histograms to see the shape of the data. A real variable does not have to be normally distributed.

In [ ]:
sns.histplot(data=apps, x="Rating", bins=20).set(
    title="App ratings", xlabel="Rating", ylabel="Number of apps")

Plots the distribution of app ratings as a histogram.

**Observations:** Most ratings are toward the high end. The lower tail is longer than the upper tail, matching the negative skew result.

In [ ]:
sns.histplot(data=apps, x="Installs_num", log_scale=True).set(
    title="App installs on a log scale", xlabel="Installs", ylabel="Number of apps")

Plots install counts on a log scale to show small and large values together.

**Observations:** Install counts vary widely and are right-skewed. The log scale makes both small and large install values easier to see.

## 9. Sampling and the Central Limit Theorem
A sample is a subset of app ratings. Repeating samples produces different sample means. A fixed seed makes the demonstration repeatable.

In [ ]:
ratings = apps["Rating"].dropna()
sample_means = []
for seed in range(500):
    sample = ratings.sample(30, random_state=seed)
    sample_means.append(sample.mean())
sample_means = pd.Series(sample_means)

Takes 500 reproducible random samples of 30 ratings and records each sample mean.

In [ ]:
sns.histplot(sample_means, bins=20).set(
    title="Means from 500 samples of 30 app ratings", xlabel="Sample mean rating", ylabel="Number of samples")

Plots the distribution of the 500 sample means.

In [ ]:
sample_mean = sample_means.mean()
sample_std = sample_means.std()

within_1 = sample_means.between(sample_mean - sample_std, sample_mean + sample_std).mean()
within_2 = sample_means.between(sample_mean - 2*sample_std, sample_mean + 2*sample_std).mean()
within_3 = sample_means.between(sample_mean - 3*sample_std, sample_mean + 3*sample_std).mean()

pd.Series({"Within 1 SD (compare with 68%)": within_1,
           "Within 2 SD (compare with 95%)": within_2,
           "Within 3 SD (compare with 99.7%)": within_3})

Measures the share of sample means within one, two, and three standard deviations.

The empirical rule expects about 68%, 95%, and 99.7% within one, two, and three standard deviations for a roughly normal distribution. Compare those reference values with the proportions above.

**Observations:** Each sample has 30 ratings and we repeat 500 times. Sample means vary less than individual ratings and form a more balanced shape. The Central Limit Theorem says sample means tend to become more normal as sample size grows, under suitable sampling conditions. These app records are not a random sample of all apps.

In [ ]:
sample_means_60 = []
for seed in range(500):
    sample = ratings.sample(60, random_state=seed)
    sample_means_60.append(sample.mean())
pd.Series(sample_means).std(), pd.Series(sample_means_60).std()

Repeats the sampling with 60 ratings and compares the variability of sample means.

The output compares spread for samples of 30 and 60. The larger sample usually has less sampling variability.

In [ ]:
two_groups = pd.Series([3] * 100 + [7] * 100)
sns.histplot(two_groups, bins=20).set(
    title="Simulated bimodal example (not app data)", xlabel="Made-up value", ylabel="Count")

Plots a clearly labelled, made-up bimodal distribution for demonstration.

**Observations:** This made-up example has two groups and two peaks. It is not from Google Play data. The real rating and install plots show skew rather than this artificial two-group example.

## 10. Compare app measures
Correlation measures how two numerical values change together. Correlation does not prove that one value causes another.

In [ ]:
apps[["Rating", "Reviews_num", "Installs_num", "Price_num", "Size_mb"]].cov()

Calculates covariance between rating, reviews, installs, price, and size.

In [ ]:
correlations = apps[["Rating", "Reviews_num", "Installs_num", "Price_num", "Size_mb"]].corr()
correlations

Calculates Pearson correlations between the selected app measures.

In [ ]:
apps[["Rating", "Reviews_num", "Installs_num", "Price_num", "Size_mb"]].corr(method="spearman")

Calculates Spearman rank correlations for the selected app measures.

In [ ]:
sns.heatmap(correlations, annot=True).set(title="Correlation between app measures")

Displays the Pearson correlation table as a heatmap.

Pearson correlation describes straight-line relationships; Spearman compares ranks. Correlations range from -1 to +1, while covariance depends on units. Read table and heatmap values to describe the strongest relationships; neither method shows cause and effect.

In [ ]:
sns.scatterplot(data=apps, x="Reviews_num", y="Installs_num").set(
    title="Reviews and installs", xlabel="Number of reviews", ylabel="Number of installs")

Plots review counts against install counts to show their relationship.

**Observations:** Compare the direction and spread in this plot with the correlation output. Very large app counts can affect the pattern; the plot shows association only, not causation.

## 11. Find and compare outliers
An outlier is an unusually small or large value. It may still be a valid app, so inspect flagged rows before changing anything.

In [ ]:
sns.boxplot(data=apps, x="Installs_num").set(
    title="Potential outliers in installs", xlabel="Number of installs", ylabel="")

Plots install counts on a log scale to show small and large values together.

The box plot shows the middle half of install counts and possible values beyond the whiskers. Popular apps can be valid high values.

In [ ]:
q1 = apps["Installs_num"].quantile(.25)
q3 = apps["Installs_num"].quantile(.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

Removes plus signs and commas, then converts install counts to numbers.

In [ ]:
iqr_outliers = apps[(apps["Installs_num"] < lower) | (apps["Installs_num"] > upper)]
len(iqr_outliers), len(iqr_outliers) / len(apps)

Removes plus signs and commas, then converts install counts to numbers.

In [ ]:
z_scores = (apps["Installs_num"] - apps["Installs_num"].mean()) / apps["Installs_num"].std()
z_outliers = apps[z_scores.abs() > 3]
len(z_outliers), len(z_outliers) / len(apps)

Removes plus signs and commas, then converts install counts to numbers.

In [ ]:
iqr_outliers[["App", "Category", "Installs", "Installs_num"]].head()

Displays example apps flagged as IQR outliers for inspection.

IQR uses quartiles; Z-score uses mean and standard deviation with a threshold of 3. They can flag different rows. Installs are skewed, so Z-scores need care. The table helps inspect whether flagged apps look like plausible popular apps.

In [ ]:
apps_removed = apps[apps["Installs_num"].between(lower, upper)]

Removes plus signs and commas, then converts install counts to numbers.

In [ ]:
apps_capped = apps.copy()
apps_capped["Installs_capped"] = apps_capped["Installs_num"].clip(lower=lower, upper=upper)

Creates a separate copy with install counts limited to the IQR boundaries.

In [ ]:
apps_retained = apps.copy()

Creates a separate copy that retains every original install value.

In [ ]:
apps["Installs_num"].agg(["mean", "median", "std", "min", "max"])

Removes plus signs and commas, then converts install counts to numbers.

In [ ]:
apps_removed["Installs_num"].agg(["mean", "median", "std", "min", "max"])

Summarizes install counts after removing flagged values.

In [ ]:
apps_capped["Installs_capped"].agg(["mean", "median", "std", "min", "max"])

Summarizes install counts after capping the extremes.

Removal and capping are only comparisons; original values remain unchanged. Since extreme installs may be real, retention is preferred for the main analysis. Removal can discard valid popular apps; capping reduces extreme influence but changes recorded values.

## 12. Check data quality

In [ ]:
apps.isna().sum()

Counts missing values in each working-data column.

In [ ]:
(apps.isna().sum() / len(apps) * 100).round(2)

Counts missing values in each working-data column.

In [ ]:
apps["Category"].value_counts().head(10)

Shows the most common app categories as a basic consistency check.

In [ ]:
((apps["Rating"] < 0) | (apps["Rating"] > 5)).sum(), (apps["Price_num"] < 0).sum()

Removes the dollar sign and converts app prices to numbers.

In [ ]:
apps_raw.duplicated().sum(), apps["App"].duplicated().sum()

Counts exact duplicate source rows and repeated app names.

In [ ]:
apps["Last_Updated"].isna().sum(), apps["App_key"].isna().sum()

Converts update dates and extracts the month as a new feature.

Missing counts and percentages measure completeness. Rating should be between 0 and 5, and price should not be negative. Duplicate app names mean app name is not a unique key. We use app names to match reviews; unmatched names can occur when an app has no review records. Missing values are reported rather than silently removed.

## 13. Conclusion
The Google Play dataset supports beginner EDA with app ratings, categories, prices, install counts, update dates, and a related review table. Ratings are mostly high, while install and review counts are strongly skewed. Summarizing reviews before joining avoids multiplying app rows. Outlier methods flag unusual values but do not prove errors, so this analysis keeps valid extreme values.

**Limitations:** App names can repeat or differ between files; some app sizes are not simple MB values; some ratings and reviews are missing; and this is observational data. Correlation does not establish cause. Check each generated table and chart when presenting the project.